# Przypisanie PS do budynków

Celem jest przygotowanie warstwy budynków, którą będzie można wykorzystać do dalszych analiz przestrzennych i wskazywania obiektów wymagających dokładniejszej obserwacji.

## Wczytanie danych

- EGMS_Thessaloniki_results.csv – punkty EGMS z cechami przemieszczeń oraz wynikami Isolation Forest i K-means;

- thessaloniki_buildings.gpkg – warstwa przestrzenna z obrysami budynków.

In [5]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

ps = pd.read_csv("EGMS_Thessaloniki_results.csv")
buildings = gpd.read_file("thessaloniki_buildings.gpkg")

print("Liczba PS:", len(ps))
print("Liczba budynków:", len(buildings))
print("CRS budynków:", buildings.crs)

Liczba PS: 5625
Liczba budynków: 21846
CRS budynków: EPSG:3035


## Usunięcie powtarzających się geometrii

In [6]:
# Usunięcie identycznych geometrii
buildings = (
    buildings
    .drop_duplicates(subset="geometry")
    .reset_index(drop=True)
    .copy()
)

# Nadanie identyfikatorów budynkom
buildings["building_id"] = np.arange(
    1, len(buildings) + 1
)

print("Liczba budynków:", len(buildings))

Liczba budynków: 21846


## Ujednolicenie układów współrzędnych

In [8]:
ps_gdf = gpd.GeoDataFrame(
    ps.copy(),
    geometry=gpd.points_from_xy(
        ps["easting"],
        ps["northing"]
    ),
    crs="EPSG:3035"
)

# Transformacja budynków do układu punktów EGMS
buildings = buildings.to_crs(ps_gdf.crs)

print("CRS PS:", ps_gdf.crs)
print("CRS budynków:", buildings.crs)

CRS PS: EPSG:3035
CRS budynków: EPSG:3035


## Wybór geometrii i atrybutów

In [9]:
building_footprints = buildings[
    ["building_id", "geometry"]
].copy()

print("Typy geometrii:")
print(building_footprints.geometry.geom_type.value_counts())

Typy geometrii:
MultiPolygon    21846
Name: count, dtype: int64


In [10]:
ps_for_join = ps_gdf[
    [
        "pid",
        "mean_velocity",
        "acceleration",
        "trend_change",
        "anomaly_score",
        "anomaly",
        "cluster",
        "cluster_name",
        "rmse_ts",
        "geometry"
    ]
].copy()

## Przypisanie najbliższego PS

Punkty EGMS w tym zbiorze leżą w regularnej siatce co 100 m (produkt Ortho, poziom L3) — każdy punkt reprezentuje komórkę 100 × 100 m, a nie konkretny budynek. Dlatego `MAX_DISTANCE = 50 m` (połowa oczka siatki): budynek dostaje punkt komórki, w której leży. Przypisany PS opisuje więc ruch terenu w otoczeniu budynku, a nie ruch samego budynku.

In [11]:
MAX_DISTANCE = 50  # [m] połowa oczka siatki EGMS Ortho (100 m)

buildings_ps_all = gpd.sjoin_nearest(
    building_footprints,
    ps_for_join,
    how="left",
    max_distance=MAX_DISTANCE,
    distance_col="distance_PS"
)

In [12]:
buildings_ps = (
    buildings_ps_all
    .reset_index(drop=True)
    .sort_values(
        ["building_id", "distance_PS", "anomaly_score"],
        ascending=[True, True, False],
        na_position="last"
    )
    .drop_duplicates(
        subset="building_id",
        keep="first"
    )
    .reset_index(drop=True)
)

buildings_ps["has_PS"] = (
    buildings_ps["pid"].notna()
)

In [13]:
matched = buildings_ps["has_PS"]

print("Wszystkie budynki:", len(buildings_ps))
print(f"Budynki z PS do {MAX_DISTANCE} m:", matched.sum())
print(f"Budynki bez PS do {MAX_DISTANCE} m:", (~matched).sum())
print("Wykorzystane unikalne PS:", buildings_ps["pid"].nunique())
print(
    "PS wewnątrz budynku:",
    (buildings_ps["distance_PS"] == 0).sum()
)
print(
    "Mediana odległości:",
    round(
        buildings_ps.loc[
            matched,
            "distance_PS"
        ].median(),
        1
    ),
    "m"
)

Wszystkie budynki: 21846
Budynki z PS do 30 m: 9196
Budynki bez PS do 30 m: 12650
Wykorzystane unikalne PS: 2885
PS wewnątrz budynku: 462
Mediana odległości: 18.7 m


## Utworzenie warstwy wynikowej

In [14]:
result = buildings.merge(
    buildings_ps.drop(
        columns=["geometry", "index_right"],
        errors="ignore"
    ),
    on="building_id",
    how="left"
)

result = gpd.GeoDataFrame(
    result,
    geometry="geometry",
    crs=buildings.crs
)

print("Budynki w wyniku:", len(result))
print("Budynki z przypisanym PS:", result["has_PS"].sum())
print("Unikalne identyfikatory:", result["building_id"].nunique())

Budynki w wyniku: 21846
Budynki z przypisanym PS: 9196
Unikalne identyfikatory: 21846


## Budynki a klastry i anomalie

Ile budynków leży w zasięgu PS z każdego klastra i ile z nich ma przypisany PS anomalny.

In [ ]:
buildings_summary = (
    result[result["has_PS"]]
    .groupby(["cluster", "cluster_name"])
    .agg(
        n_buildings=("building_id", "count"),
        n_buildings_anomaly=("anomaly", "sum"),
        median_velocity=("mean_velocity", "median")
    )
)

buildings_summary["percent_anomaly"] = (
    buildings_summary["n_buildings_anomaly"]
    / buildings_summary["n_buildings"]
    * 100
).round(1)

print("Budynki z PS anomalnym:", int(result["anomaly"].fillna(0).sum()))
buildings_summary

In [15]:
result.to_file(
    "budynki_EGMS_results.gpkg",
    layer="budynki_EGMS",
    driver="GPKG"
)

# Ta sama tabela bez geometrii — do oddania jako CSV
result.drop(columns="geometry").to_csv(
    "budynki_EGMS_results.csv",
    index=False,
    encoding="utf-8"
)

print("Zapisano: budynki_EGMS_results.gpkg, budynki_EGMS_results.csv")

Zapisano: budynki_EGMS_results.gpkg
